# CPU bootstrap，不启动 actor

这一步只在 CPU 会话准备 `xfuser==0.4.5` 和 `yunchang==0.6.4`。它不下载权重，不替换已经安装的 torch / torchvision / ray，也不克隆 ComfyUI。

`comfy` 和 `folder_paths` 缺失不会挡住 actor admission。四个权重 MISSING 也不会。`xfuser` 或 `yunchang` 缺失会挡住。

CPU torch 上导入 `xfuser.core.distributed` 会抛出 `NotImplementedError: No Accelerators available`。那是 xfuser 自己的加速器检查，不是缺模块。此时期望 `materials_ready_for_gpu_stage` 为 true，`gpu_stage_permitted` 和 `actor_admission_passed` 为 false。不要在这个会话开 GPU actor。

代码放在 Kaggle dataset input 里。No persistence 会清掉 /tmp 和 working 里的安装结果，所以 GPU 会话要重新跑一遍 plan/apply/check，再跑 6 分钟的 actor-admission。


In [ ]:
import os, sys
from pathlib import Path

def find_root():
    seeds = [Path.cwd(), Path("/kaggle/working"), Path("/kaggle/input")]
    seen = []
    for seed in seeds:
        if seed.exists():
            seen.extend([seed, *list(seed.glob("*")), *list(seed.glob("*/*"))])
    for candidate in seen:
        if (candidate / "scripts" / "bootstrap_actor_admission.py").is_file():
            return candidate
    raise SystemExit("把本文件夹作为 Kaggle input dataset 挂上，不要只依赖 /tmp。")

ROOT = find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
sys.path.insert(0, str(ROOT / "src"))
REPORT = Path("/kaggle/working/h3-bootstrap")
REPORT.mkdir(parents=True, exist_ok=True)
print({"root": str(ROOT), "report": str(REPORT), "actor_admission_passed": False})


In [ ]:
import json, subprocess, sys
plan = subprocess.run([sys.executable, "scripts/bootstrap_actor_admission.py", "--stage", "plan", "--report", str(REPORT / "plan.json")], check=False)
print("plan", plan.returncode)
if plan.returncode != 0:
    raise SystemExit("pip plan 拒绝执行。通常是它会改动 torch。不要 apply。")
apply = subprocess.run([sys.executable, "scripts/bootstrap_actor_admission.py", "--stage", "apply", "--report", str(REPORT / "apply.json")], check=False)
print("apply", apply.returncode)
check = subprocess.run([sys.executable, "scripts/bootstrap_actor_admission.py", "--stage", "check", "--report", str(REPORT / "check.json")], check=False)
report = json.loads((REPORT / "check.json").read_text(encoding="utf-8"))
print(json.dumps({
    "check": check.returncode,
    "materials_ready_for_gpu_stage": report["materials_ready_for_gpu_stage"],
    "gpu_stage_permitted": report["gpu_stage_permitted"],
    "actor_admission_passed": report["actor_admission_passed"],
    "xfuser": report["required_imports"]["xfuser.core.distributed"]["status"],
    "yunchang": report["required_imports"]["yunchang"]["status"],
    "comfy": report["not_required_for_actor_admission"]["comfy"]["status"],
    "torch": report["installed_versions"]["torch"],
}, ensure_ascii=False))
if report["gpu_stage_permitted"]:
    raise SystemExit("这个 CPU 笔记本不允许 gpu_stage_permitted。")
print("下载 /kaggle/working/h3-bootstrap 的三个 JSON。GPU 会话另开，先重复这三步，device_count 为 2 后再跑 actor-admission，预算 6 分钟。")
